# Practical 7 — Logistic Regression from Scratch

**Lectures:** 14. Logistic Regression Part I · 15. Logistic Regression Part II

Logistic regression models the probability of the positive class with the **sigmoid** of a linear function:

$$h_\theta(\mathbf x) = \sigma(\boldsymbol\theta^\top\mathbf x) = \frac{1}{1+e^{-\boldsymbol\theta^\top\mathbf x}},\qquad
\log\frac{h_\theta(\mathbf x)}{1-h_\theta(\mathbf x)} = \boldsymbol\theta^\top\mathbf x\;\;\text{(the log-odds are linear)}$$

It is trained by minimising the **cross-entropy (log-loss)** cost

$$J(\boldsymbol\theta) = -\frac1m\sum_{i=1}^m\Big[y^{(i)}\log h_\theta(\mathbf x^{(i)}) + (1-y^{(i)})\log\big(1-h_\theta(\mathbf x^{(i)})\big)\Big] + \frac{\lambda}{2m}\sum_{j\ge1}\theta_j^2,$$

whose gradient has exactly the same form as for linear regression:
$\nabla J = \frac1m X^\top(\sigma(X\boldsymbol\theta)-\mathbf y) + \frac{\lambda}{m}[0,\theta_1,\dots,\theta_n]^\top$.

We use the **banknote authentication** dataset: four statistics of wavelet-transformed images of banknotes
(`Variance`, `Skewness`, `Curtosis`, `Entropy`), with `Class` = 1 for forged notes.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

bank = pd.read_csv("https://raw.githubusercontent.com/lshi/ml/main/data/bill_authentication.csv")
print(bank.head(), "\n", bank["Class"].value_counts())
X = bank.drop(columns="Class").to_numpy(); y = bank["Class"].to_numpy()
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y, random_state=0)
sc = StandardScaler().fit(X_tr)
X_tr_s, X_te_s = sc.transform(X_tr), sc.transform(X_te)

def add_bias(X):
    return np.column_stack([np.ones(len(X)), X])

## 1. Sigmoid, odds and log-odds (≈10 min)
**TODO 1.1** Implement `sigmoid(z)`. Plot it on $[-8, 8]$. Then fill in the table: for probabilities
$p \in \{0.1, 0.25, 0.5, 0.75, 0.9\}$ compute the **odds** $p/(1-p)$ and **log-odds**, and check that
`sigmoid(log_odds)` gives back $p$.

In [ ]:
def sigmoid(z):
    # YOUR CODE HERE
    raise NotImplementedError

# YOUR CODE HERE

## 2. Training by gradient descent (≈30 min)
**TODO 2.1** Implement the cost and gradient (with optional L2 regularisation — do **not** penalise the
bias) and a gradient-descent training loop. Clip predicted probabilities to $[10^{-12}, 1-10^{-12}]$ inside
the cost to avoid $\log 0$.

In [ ]:
def cost(theta, Xb, y, lam=0.0):
    # YOUR CODE HERE
    raise NotImplementedError

def gradient(theta, Xb, y, lam=0.0):
    # YOUR CODE HERE
    raise NotImplementedError

def train_logreg(Xb, y, alpha=0.5, n_iters=3000, lam=0.0):
    theta = np.zeros(Xb.shape[1]); history = []
    for _ in range(n_iters):
        # YOUR CODE HERE
        history.append(cost(theta, Xb, y, lam))
    return theta, np.array(history)

def predict_proba(theta, X):
    return sigmoid(add_bias(X) @ theta)

**TODO 2.2 — Check your gradient numerically.** A wrong gradient is the most common bug. Compare your
`gradient` with a centred finite difference $\frac{J(\theta+\epsilon e_j)-J(\theta-\epsilon e_j)}{2\epsilon}$
at a random $\boldsymbol\theta$; the maximum absolute difference should be tiny (< 1e-6).

In [ ]:
# YOUR CODE HERE

## 3. A 2-D model and its decision boundary (≈20 min)
**TODO 3.1** Train on the first two (standardised) features only, `Variance` and `Skewness`. Plot the cost
history, then plot the training data coloured by class together with the **decision boundary**
$\theta_0 + \theta_1x_1 + \theta_2x_2 = 0$ (i.e. $h_\theta = 0.5$). Also draw the $h_\theta = 0.1$ and $0.9$ contours.

In [ ]:
# YOUR CODE HERE

## 4. All features; comparison with scikit-learn; interpretation (≈20 min)
**TODO 4.1** Train on all four features. Compare your coefficients and test accuracy with
`LogisticRegression(C=1e6)` (a huge `C` means almost no regularisation; scikit-learn's `C` is $1/\lambda$).
Then compute the **odds ratios** $e^{\theta_j}$. Interpret one of them in words. Why might your coefficients differ from scikit-learn's even though the accuracies are almost identical?

In [ ]:
from sklearn.linear_model import LogisticRegression

# YOUR CODE HERE

*Write your answer here.*

## 5. Regularisation (≈15 min)
**TODO 5.1** The banknote data are nearly linearly separable, so without regularisation the coefficients grow
as long as you keep training. Train with $\lambda \in \{0, 0.1, 1, 10, 100\}$ for 5000 iterations and tabulate the
coefficient norm $\lVert\theta_{1:n}\rVert$, the training log-loss and the test accuracy. What happens as $\lambda$ grows?

In [ ]:
# YOUR CODE HERE

## 6. Multinomial (softmax) regression (≈25 min)
For $K$ classes, keep one parameter vector per class ($\Theta\in\mathbb R^{(n+1)\times K}$) and use the
**softmax**: $P(y=k\mid\mathbf x) = \dfrac{e^{\boldsymbol\theta_k^\top\mathbf x}}{\sum_j e^{\boldsymbol\theta_j^\top\mathbf x}}$.
With one-hot labels $Y$, the cross-entropy gradient is $\frac1m X^\top(\operatorname{softmax}(X\Theta) - Y)$.

**TODO 6.1** Implement `softmax` (subtract the row maximum before exponentiating, for numerical stability) and train
softmax regression by gradient descent on the standardised **iris** data. Report test accuracy and compare with
`LogisticRegression()` (which uses the multinomial loss by default).

In [ ]:
from sklearn.datasets import load_iris

def softmax(Z):
    # YOUR CODE HERE
    raise NotImplementedError

def train_softmax(Xb, y, K, alpha=0.5, n_iters=3000):
    Y = np.eye(K)[y]                         # one-hot labels
    Theta = np.zeros((Xb.shape[1], K))
    for _ in range(n_iters):
        # YOUR CODE HERE
        raise NotImplementedError
    return Theta

iris = load_iris()
Xi_tr, Xi_te, yi_tr, yi_te = train_test_split(iris.data, iris.target, test_size=0.3, stratify=iris.target, random_state=0)
si = StandardScaler().fit(Xi_tr)
# YOUR CODE HERE

## 7. Extension: non-linear decision boundaries
Logistic regression is linear in its features — but we can add polynomial features (as in Practical 4).
On `sklearn.datasets.make_moons(n_samples=400, noise=0.25, random_state=0)`, compare the decision boundaries and
test accuracy of logistic regression with degree 1 and degree 3 polynomial features, with and without regularisation.

In [ ]:
# YOUR CODE HERE

## Lab report checklist (Practical 7)
1. Your gradient check result, and the decision-boundary figure for the 2-feature model.
2. A table comparing your coefficients and test accuracy with scikit-learn's; the odds ratios and your interpretation of one of them.
3. The regularisation table and an explanation of why unregularised logistic regression misbehaves on (nearly) separable data.
4. Softmax regression test accuracy on iris versus scikit-learn.
5. Optionally, the test-set confusion matrix / precision / recall of your banknote model (reuse your Practical 6 functions).